# Rule Engine Eval — Can the rules tell good tables from bad ones?

Uses the 500 pre-extracted tables in `eval_unitable_results/`.
No model inference needed — just loads `_pred.html` and `_gt.html`.

**Logic:**
- **Ground truth label** = TEDS (full) of pred vs gt  
  - `TEDS ≥ GOOD_THRESHOLD` → table is *actually good*
  - `TEDS < GOOD_THRESHOLD` → table is *actually bad*
- **Rule engine label** = tier on pred HTML
  - `HIGH` → rule engine thinks it's *good*
  - `MEDIUM / LOW` → rule engine thinks it's *bad*

From there we compute: TP, TN, FP (over-flagged), FN (missed problems).

In [15]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

project_root = Path.cwd().parent
huma_root    = project_root / 'Huma-Huma'

for p in [project_root, huma_root, project_root / 'pubtabnet' / 'src']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from shared import rule_engine, table_validator
from metric import TEDS

EVAL_DIR       = project_root / 'eval_unitable_results'
GOOD_THRESHOLD = 0.5   # TEDS (full) >= this → table is 'actually good'

teds_full = TEDS(structure_only=False)
wrap      = lambda h: f'<html><body>{h}</body></html>'

print('Imports OK')

Imports OK


In [16]:
# ── Load pred/gt pairs ────────────────────────────────────────────────────────
SAMPLE_N = 10  # set to None to run all 500

pairs = []
for pred_path in sorted(EVAL_DIR.glob('*_pred.html')):
    stem    = pred_path.stem.replace('_pred', '')
    gt_path = EVAL_DIR / f'{stem}_gt.html'
    if gt_path.exists():
        pairs.append((stem, pred_path, gt_path))

if SAMPLE_N:
    pairs = pairs[:SAMPLE_N]

print(f'Running on {len(pairs)} pairs')

Running on 10 pairs


In [17]:
# ── Run rule engine + TEDS on every pair ──────────────────────────────────────
rows = []
rules_data = rule_engine.load_rules()

for stem, pred_path, gt_path in pairs:
    pred_html = pred_path.read_text()
    gt_html   = gt_path.read_text()

    # Rule engine on prediction
    re_result  = rule_engine.evaluate(pred_html, table_id=stem, rules_data=rules_data)
    val_result = table_validator.validate_table(pred_html, table_id=stem)

    # TEDS (full): how close is the prediction to ground truth?
    teds_score = teds_full.evaluate(wrap(pred_html), wrap(gt_html))

    # Ground truth label
    actually_good = teds_score >= GOOD_THRESHOLD

    # Rule engine label (HIGH = good, anything else = bad)
    re_says_good  = re_result.tier  == 'HIGH'
    val_says_good = val_result.tier == 'HIGH'

    rows.append({
        'image':          stem,
        'teds_full':      round(teds_score, 4),
        'actually_good':  actually_good,
        're_tier':        re_result.tier,
        're_confidence':  re_result.confidence,
        're_says_good':   re_says_good,
        're_fired':       ', '.join(re_result.fired_rules) or 'none',
        'val_tier':       val_result.tier,
        'val_confidence': val_result.confidence,
        'val_says_good':  val_says_good,
        'val_failures':   ', '.join(val_result.failure_modes) or 'none',
    })

df = pd.DataFrame(rows)
print(f'Processed {len(df)} tables')
print(f"Actually good (TEDS ≥ {GOOD_THRESHOLD}): {df['actually_good'].sum()} / {len(df)}")
print(f"TEDS distribution:")
print(df['teds_full'].describe().round(3))

Processed 10 tables
Actually good (TEDS ≥ 0.5): 9 / 10
TEDS distribution:
count    10.000
mean      0.671
std       0.113
min       0.460
25%       0.626
50%       0.642
75%       0.752
max       0.848
Name: teds_full, dtype: float64


In [18]:
# ── Confusion matrix helper ───────────────────────────────────────────────────
def confusion(df, says_good_col, label=''):
    tp = ((~df[says_good_col]) & (~df['actually_good'])).sum()  # flagged bad, actually bad
    tn = (( df[says_good_col]) & ( df['actually_good'])).sum()  # said good,  actually good
    fp = ((~df[says_good_col]) & ( df['actually_good'])).sum()  # flagged bad, actually good  ← false alarm
    fn = (( df[says_good_col]) & (~df['actually_good'])).sum()  # said good,  actually bad   ← missed
    total = len(df)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    accuracy  = (tp + tn) / total

    print(f'\n── {label} ──')
    print(f'  TP (caught bad tables):        {tp:4d}')
    print(f'  TN (passed good tables):       {tn:4d}')
    print(f'  FP (false alarm on good table):{fp:4d}  ← over-flagging')
    print(f'  FN (missed bad table):         {fn:4d}  ← missed problems')
    print(f'  Precision: {precision:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}  Accuracy: {accuracy:.3f}')
    return {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
            'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}

re_stats  = confusion(df, 're_says_good',  label='Rule Engine')
val_stats = confusion(df, 'val_says_good', label='Table Validator')


── Rule Engine ──
  TP (caught bad tables):           0
  TN (passed good tables):          8
  FP (false alarm on good table):   1  ← over-flagging
  FN (missed bad table):            1  ← missed problems
  Precision: 0.000  Recall: 0.000  F1: 0.000  Accuracy: 0.800

── Table Validator ──
  TP (caught bad tables):           0
  TN (passed good tables):          7
  FP (false alarm on good table):   2  ← over-flagging
  FN (missed bad table):            1  ← missed problems
  Precision: 0.000  Recall: 0.000  F1: 0.000  Accuracy: 0.700


In [19]:
# ── TEDS distribution by tier ─────────────────────────────────────────────────
print('Mean TEDS (full) by Rule Engine tier:')
display(df.groupby('re_tier')['teds_full'].describe().round(3))

print('\nMean TEDS (full) by Table Validator tier:')
display(df.groupby('val_tier')['teds_full'].describe().round(3))

Mean TEDS (full) by Rule Engine tier:


,count,mean,std,min,25%,50%,75%,max
re_tier,,,,,,,,
HIGH,9.0,0.673,0.119,0.460,0.624,0.634,0.785,0.848
LOW,1.0,0.651,NaN,0.651,0.651,0.651,0.651,0.651



Mean TEDS (full) by Table Validator tier:


,count,mean,std,min,25%,50%,75%,max
val_tier,,,,,,,,
HIGH,8.0,0.676,0.127,0.460,0.624,0.632,0.789,0.848
LOW,2.0,0.651,0.001,0.651,0.651,0.651,0.651,0.652


In [20]:
# ── Which rules fire most on good vs bad tables ───────────────────────────────
import re as re_mod
from collections import Counter

rule_ids = [f'R{str(i).zfill(3)}' for i in range(1, 12)]

good_tables = df[df['actually_good']]
bad_tables  = df[~df['actually_good']]

def rule_fire_rates(subset, label):
    n = len(subset)
    counts = Counter()
    for fired in subset['re_fired']:
        for r in fired.split(', '):
            if r != 'none':
                counts[r] += 1
    print(f'\n{label} (n={n}) — rule fire rates:')
    for rid in sorted(counts, key=lambda x: -counts[x]):
        print(f'  {rid}: {counts[rid]:4d} / {n}  ({counts[rid]/n*100:.1f}%)')
    if not counts:
        print('  (no rules fired)')

rule_fire_rates(good_tables, 'Actually GOOD tables')
rule_fire_rates(bad_tables,  'Actually BAD tables')


Actually GOOD tables (n=9) — rule fire rates:
  R002:    3 / 9  (33.3%)
  R004:    1 / 9  (11.1%)

Actually BAD tables (n=1) — rule fire rates:
  R002:    1 / 1  (100.0%)


In [21]:
# ── False positives: rule flagged as bad, but TEDS says good ──────────────────
fp_df = df[(~df['re_says_good']) & df['actually_good']].sort_values('teds_full', ascending=False)
print(f'False positives (over-flagged): {len(fp_df)}')
display(fp_df[['image', 'teds_full', 're_tier', 're_confidence', 're_fired']].head(20))

False positives (over-flagged): 1


,image,teds_full,re_tier,re_confidence,re_fired
9,imgid_551906_TEDS0.671,0.6508,LOW,54,"R002, R004"


In [22]:
# ── False negatives: rule said good, but TEDS says bad ───────────────────────
fn_df = df[df['re_says_good'] & (~df['actually_good'])].sort_values('teds_full')
print(f'False negatives (missed problems): {len(fn_df)}')
display(fn_df[['image', 'teds_full', 're_tier', 're_confidence', 're_fired']].head(20))

False negatives (missed problems): 1


,image,teds_full,re_tier,re_confidence,re_fired
3,imgid_549778_TEDS0.600,0.4599,HIGH,88,R002


In [23]:
# ── Inspect a specific false negative ────────────────────────────────────────
# Rule said HIGH (good) but TEDS is low — what does the table look like?
if len(fn_df) > 0:
    row      = fn_df.iloc[0]
    pred_html = (EVAL_DIR / f"{row['image']}_pred.html").read_text()
    gt_html   = (EVAL_DIR / f"{row['image']}_gt.html").read_text()

    print(f"Image: {row['image']}")
    print(f"TEDS: {row['teds_full']}  |  RE tier: {row['re_tier']}  |  Fired: {row['re_fired']}")

    style = '<style>table{border-collapse:collapse;font-size:11px} td,th{border:1px solid #ccc;padding:3px 6px}</style>'
    display(HTML(f"""{style}
    <div style='display:flex;gap:30px'>
      <div><h4>Prediction</h4>{pred_html}</div>
      <div><h4>Ground Truth</h4>{gt_html}</div>
    </div>"""))

Image: imgid_549778_TEDS0.600
TEDS: 0.4599  |  RE tier: HIGH  |  Fired: R002


In [24]:
# ── Inspect a specific false positive ────────────────────────────────────────
# Rule flagged as bad but TEDS says it's actually fine — why did it fire?
if len(fp_df) > 0:
    row       = fp_df.iloc[0]
    pred_html = (EVAL_DIR / f"{row['image']}_pred.html").read_text()
    gt_html   = (EVAL_DIR / f"{row['image']}_gt.html").read_text()

    print(f"Image: {row['image']}")
    print(f"TEDS: {row['teds_full']}  |  RE tier: {row['re_tier']}  |  Fired: {row['re_fired']}")

    # Also show rule engine features for diagnosis
    feat = rule_engine.extract_features(pred_html)
    print('\nFeatures:')
    for k, v in feat.items():
        print(f'  {k}: {v}')

    style = '<style>table{border-collapse:collapse;font-size:11px} td,th{border:1px solid #ccc;padding:3px 6px}</style>'
    display(HTML(f"""{style}
    <div style='display:flex;gap:30px'>
      <div><h4>Prediction</h4>{pred_html}</div>
      <div><h4>Ground Truth</h4>{gt_html}</div>
    </div>"""))

Image: imgid_551906_TEDS0.671
TEDS: 0.6508  |  RE tier: LOW  |  Fired: R002, R004

Features:
  num_header_rows: 1
  num_body_rows: 16
  all_header_cells_empty: False
  modal_col_consistency: 0.529
  empty_cell_ratio: 0.0
  duplicate_body_row_ratio: 0.0
  ocr_artifact_ratio: 0.0
  numeric_garble_ratio: 0.0
  whitespace_cell_ratio: 0.0
  single_col_row_fraction: 0.562
  max_cell_length: 113
  max_colspan: 1
  max_rowspan: 5


In [25]:
# ── Full results table ────────────────────────────────────────────────────────
display(df.sort_values('teds_full'))

,image,teds_full,actually_good,re_tier,re_confidence,re_says_good,re_fired,val_tier,val_confidence,val_says_good,val_failures
3,imgid_549778_TEDS0.600,0.4599,False,HIGH,88,True,R002,HIGH,88,True,column_consistency
6,imgid_551440_TEDS0.894,0.6232,True,HIGH,100,True,none,HIGH,100,True,none
2,imgid_549482_TEDS0.855,0.6242,True,HIGH,100,True,none,HIGH,100,True,none
1,imgid_549416_TEDS0.833,0.6299,True,HIGH,88,True,R002,HIGH,88,True,column_consistency
0,imgid_548924_TEDS0.911,0.6336,True,HIGH,100,True,none,HIGH,100,True,none
9,imgid_551906_TEDS0.671,0.6508,True,LOW,54,False,"R002, R004",LOW,54,False,"column_consistency, single_col_collapse"
8,imgid_551628_TEDS0.787,0.6517,True,HIGH,88,True,R002,LOW,54,False,"column_consistency, empty_cell_ratio, duplicat..."
4,imgid_549838_TEDS0.865,0.7852,True,HIGH,100,True,none,HIGH,100,True,none
5,imgid_549888_TEDS0.825,0.8023,True,HIGH,100,True,none,HIGH,100,True,none
7,imgid_551593_TEDS0.894,0.8483,True,HIGH,100,True,none,HIGH,100,True,none
